# Contoso RH - Bronze, Silver e Gold
Notebook oficial do Lab 03. Anexe `ContosoRH_Lakehouse` como Lakehouse padrao, carregue os 10 CSVs em `Files` e execute **Run all**.

Saidas de sucesso: `LAB03_BRONZE_OK`, `LAB03_SILVER_OK` e `LAB03_GOLD_VALIDATION_OK`.

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql import types as T

seed_path = "Files"
batch_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

for schema_name in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

schemas = {
    "colaboradores": T.StructType([T.StructField("Matricula", T.StringType()), T.StructField("Nome", T.StringType()), T.StructField("CPF", T.StringType()), T.StructField("Cargo", T.StringType()), T.StructField("Departamento", T.StringType()), T.StructField("Pais", T.StringType()), T.StructField("DataAdmissao", T.DateType()), T.StructField("Salario", T.DoubleType()), T.StructField("Status", T.StringType())]),
    "ferias": T.StructType([T.StructField("FeriaId", T.StringType()), T.StructField("Matricula", T.StringType()), T.StructField("DataInicio", T.DateType()), T.StructField("DataFim", T.DateType()), T.StructField("DiasGozados", T.IntegerType()), T.StructField("SaldoDias", T.IntegerType()), T.StructField("Status", T.StringType())]),
    "tickets": T.StructType([T.StructField("TicketId", T.StringType()), T.StructField("Matricula", T.StringType()), T.StructField("Categoria", T.StringType()), T.StructField("Prioridade", T.StringType()), T.StructField("DataAbertura", T.DateType()), T.StructField("DataResolucao", T.DateType()), T.StructField("Status", T.StringType())]),
    "beneficios": T.StructType([T.StructField("BeneficioId", T.StringType()), T.StructField("Matricula", T.StringType()), T.StructField("Tipo", T.StringType()), T.StructField("Valor", T.DoubleType()), T.StructField("DataAtivacao", T.DateType()), T.StructField("Ativo", T.BooleanType())]),
    "holerites": T.StructType([T.StructField("HoleriteId", T.StringType()), T.StructField("Matricula", T.StringType()), T.StructField("Competencia", T.StringType()), T.StructField("SalarioBruto", T.DoubleType()), T.StructField("Descontos", T.DoubleType()), T.StructField("SalarioLiquido", T.DoubleType()), T.StructField("DataPagamento", T.DateType())]),
    "irrf": T.StructType([T.StructField("IrrfId", T.StringType()), T.StructField("Matricula", T.StringType()), T.StructField("Competencia", T.StringType()), T.StructField("BaseCalculo", T.DoubleType()), T.StructField("Aliquota", T.DoubleType()), T.StructField("ValorRetido", T.DoubleType()), T.StructField("DeducaoDependentes", T.DoubleType())]),
    "encargos": T.StructType([T.StructField("EncargoId", T.StringType()), T.StructField("Matricula", T.StringType()), T.StructField("Competencia", T.StringType()), T.StructField("Tipo", T.StringType()), T.StructField("BaseCalculo", T.DoubleType()), T.StructField("Valor", T.DoubleType())]),
    "provisoes": T.StructType([T.StructField("ProvisaoId", T.StringType()), T.StructField("Matricula", T.StringType()), T.StructField("Tipo", T.StringType()), T.StructField("Competencia", T.StringType()), T.StructField("ValorProvisionado", T.DoubleType()), T.StructField("Acumulado", T.DoubleType())]),
    "reembolsos": T.StructType([T.StructField("ReembolsoId", T.StringType()), T.StructField("Matricula", T.StringType()), T.StructField("Categoria", T.StringType()), T.StructField("Valor", T.DoubleType()), T.StructField("Moeda", T.StringType()), T.StructField("DataDespesa", T.DateType()), T.StructField("Status", T.StringType())]),
    "aprovacoes": T.StructType([T.StructField("AprovacaoId", T.StringType()), T.StructField("ReembolsoId", T.StringType()), T.StructField("AprovadorMatricula", T.StringType()), T.StructField("Nivel", T.IntegerType()), T.StructField("Decisao", T.StringType()), T.StructField("DataDecisao", T.DateType()), T.StructField("Comentario", T.StringType())]),
}
expected_counts = {"colaboradores": 12, "ferias": 12, "tickets": 12, "beneficios": 13, "holerites": 14, "irrf": 12, "encargos": 14, "provisoes": 13, "reembolsos": 13, "aprovacoes": 12}
primary_keys = {"colaboradores": "Matricula", "ferias": "FeriaId", "tickets": "TicketId", "beneficios": "BeneficioId", "holerites": "HoleriteId", "irrf": "IrrfId", "encargos": "EncargoId", "provisoes": "ProvisaoId", "reembolsos": "ReembolsoId", "aprovacoes": "AprovacaoId"}
print(f"Configuracao pronta: seed_path={seed_path}, batch_id={batch_id}")

In [ ]:
bronze_counts = {}
for table_name, table_schema in schemas.items():
    source = f"{seed_path}/{table_name}.csv"
    dataframe = (spark.read.option("header", True).option("dateFormat", "yyyy-MM-dd").schema(table_schema).csv(source)
        .withColumn("_source_file", F.input_file_name())
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_batch_id", F.lit(batch_id)))
    row_count = dataframe.count()
    if row_count != expected_counts[table_name]:
        raise ValueError(f"{table_name}: esperado {expected_counts[table_name]} linhas, recebido {row_count}")
    dataframe.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"bronze.{table_name}")
    bronze_counts[table_name] = row_count

print("Bronze:", bronze_counts)
print("LAB03_BRONZE_OK")

In [ ]:
silver_counts = {}
for table_name, key_column in primary_keys.items():
    dataframe = spark.table(f"bronze.{table_name}").drop("_source_file", "_ingested_at", "_batch_id").dropDuplicates()
    for field in dataframe.schema.fields:
        if isinstance(field.dataType, T.StringType):
            dataframe = dataframe.withColumn(field.name, F.trim(F.col(field.name)))
    dataframe = dataframe.filter(F.col(key_column).isNotNull() & (F.col(key_column) != "")).dropDuplicates([key_column])
    dataframe.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"silver.{table_name}")
    silver_counts[table_name] = dataframe.count()

if silver_counts != expected_counts:
    raise ValueError(f"Contagens Silver divergentes: {silver_counts}")

colaboradores = spark.table("silver.colaboradores").select("Matricula")
for table_name in ["ferias", "tickets", "beneficios", "holerites", "irrf", "encargos", "provisoes", "reembolsos"]:
    orphan_count = spark.table(f"silver.{table_name}").join(colaboradores, "Matricula", "left_anti").count()
    if orphan_count:
        raise ValueError(f"{table_name}: {orphan_count} Matricula(s) orfa(s)")

reembolsos = spark.table("silver.reembolsos").select("ReembolsoId")
if spark.table("silver.aprovacoes").join(reembolsos, "ReembolsoId", "left_anti").count():
    raise ValueError("aprovacoes: ReembolsoId orfao")

print("Silver:", silver_counts)
print("LAB03_SILVER_OK")

In [ ]:
dim_colaborador = (spark.table("silver.colaboradores")
    .select(F.col("Matricula").alias("colaborador_id"), "Matricula", "Nome", "Cargo", "Departamento", "Pais", "DataAdmissao", "Status")
    .dropDuplicates(["colaborador_id"]))

folha = spark.table("silver.holerites")
dim_tempo = (folha.select(F.col("Competencia").alias("ano_mes"), F.col("DataPagamento").alias("data"))
    .groupBy("ano_mes").agg(F.min("data").alias("data"))
    .withColumn("ano", F.year("data"))
    .withColumn("mes", F.month("data"))
    .withColumn("data_id", F.col("data")))

fact_folha = (folha.select(
    F.concat_ws("|", F.col("HoleriteId"), F.col("Matricula"), F.col("Competencia")).alias("folha_id"),
    F.col("Matricula").alias("colaborador_id"),
    F.col("DataPagamento").alias("data"),
    F.col("Competencia"),
    F.col("Competencia").alias("ano_mes"),
    F.col("SalarioBruto").cast("double").alias("salario_bruto"),
    F.col("Descontos").cast("double").alias("descontos"),
    F.col("SalarioLiquido").cast("double").alias("salario_liquido"))
    .withColumn("data_id", F.col("data")))

dim_colaborador.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_colaborador")
dim_tempo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_tempo")
fact_folha.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fact_folha_pagamento")

In [ ]:
gold_colab = spark.table("gold.dim_colaborador")
gold_tempo = spark.table("gold.dim_tempo")
gold_folha = spark.table("gold.fact_folha_pagamento")

checks = {
    "dim_colaborador_rows": gold_colab.count() == 12,
    "dim_tempo_rows": gold_tempo.count() == 2,
    "fact_folha_rows": gold_folha.count() == 14,
    "colaborador_unique": gold_colab.select("colaborador_id").distinct().count() == gold_colab.count(),
    "folha_unique": gold_folha.select("folha_id").distinct().count() == gold_folha.count(),
    "keys_not_null": gold_folha.filter("folha_id IS NULL OR colaborador_id IS NULL OR ano_mes IS NULL").count() == 0,
    "employee_relationship": gold_folha.join(gold_colab, "colaborador_id", "left_anti").count() == 0,
    "period_relationship": gold_folha.join(gold_tempo, "ano_mes", "left_anti").count() == 0,
    "pii_not_in_gold": "CPF" not in gold_colab.columns and "Salario" not in gold_colab.columns,
    "graph_money_is_double": all(dict(gold_folha.dtypes)[column] == "double" for column in ["salario_bruto", "descontos", "salario_liquido"]),
}

failed = [name for name, passed in checks.items() if not passed]
print("Gold checks:", checks)
if failed:
    raise ValueError(f"Validacao Gold falhou: {failed}")

display(gold_colab.orderBy("colaborador_id"))
display(gold_folha.orderBy("ano_mes", "colaborador_id"))
print("LAB03_GOLD_VALIDATION_OK")